# 時系列・外部プレイヤー検証：データ準備と監査（学習なし）

対象：しょぼんｎ → ウルトラ立直。当該試合終了後の段位区分分類です。
元ノートブックは変更せず、23特徴量の式も維持します。
**このノートブックにはLightGBM学習、Optuna、牌譜解析、最終性能評価を含めません。**

基準ソース：`yuukienomoto/report_-syobon_n` commit `4d7b41dae7901b08ef7f865b8591348f2cbed642`。
履歴値：基準版 ROC-AUC **0.8340**、外部 Accuracy **35.59%**（Balanced Accuracy **50.80%**）。
再計算した結果ではなく、比較用の保存値です。

実行順：**セル1 → セル2を設定 → セル3 → セル4 → セル5 → セル6**。
不足や不確実性があれば表とCSVを出して停止します。入力を直したらセル3から再実行してください。
この段階で作るのは学習可能性を検査した入力行列であり、性能の改善を保証するものではありません。


## 入力データの契約（牌譜再解析は自動実行しません）

各プレイヤーについて2表を用意します。CSVまたはParquet、または明示的に指定したDataFrameを使用可能です。

|表|必須列|条件|
|---|---|---|
|manifest|`file_id`, `dan`, `game_order`|本人の全有効対局を段位17・初期期間も含めて列挙。`game_order`は初回からの1始まり連番。段位列名は設定で変更可|
|features|`file_id` と基本18特徴量|**補完・平滑化前**の値。非発生事象のNaNは保持。既存の補完済み`micro_dataset_df`は使用不可|

`micro_features_df`（しょぼんn）と`ultra_micro_features_df`（外部）は再利用候補です。
ただし前者は従来の6,015試合だけなので、段位17等の履歴が不足する可能性があります。
manifestを抽出済み特徴量の行だけから作ると、欠落した試合を検出できません。
`model_target_df` / 評価用`ultra_target_df`も全対局台帳の代わりにはなりません。

既存Colabのメモリに残っていれば、ノートブックを保存変更せず一時セルで例えば
`micro_features_df.to_csv('/content/drive/MyDrive/卒論/validation_inputs/shobon_raw.csv', index=False)`
と保存して再利用できます（保存先フォルダを先に作成）。すでに保存済みならそのファイルを指定します。
全対局台帳の段位は`dan_map`や全件の`dan_df`が候補ですが、**時系列と完全性は別途確認が必要**です。
ZIPやXMLはこのノートブックでは読みません。

対局IDの日時は通常「時」単位です。同一時間内のハッシュ順を真の対局順とみなしません。
`game_order`の順序根拠と、全対局台帳の完全性・元特徴量であることを確認した場合にだけ、セル2の確認フラグをTrueにしてください。
単に検査を通すためのフラグ変更は不可です。証拠がなければ監査で停止するのが正常です。


## セル1：Google Drive接続


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## セル2：入力設定（ここを編集）


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import json, re, hashlib, uuid
from datetime import datetime, timezone
from IPython.display import display

ROOT = Path('/content/drive/MyDrive/卒論')
REPORT_ROOT = ROOT / 'validation_reports'
WINDOW = 10
INITIAL_EXCLUDE = 1500

SOURCES = {
    'shobon': {
        'manifest_path': ROOT / 'validation_inputs/shobon_manifest.csv',
        'features_path': ROOT / 'validation_inputs/shobon_raw.csv',
        'dan_column': 'dan',  # 保存データがshobon_danなら変更
        'raw_features_confirmed': False,
        'full_history_confirmed': False,
        'order_verified': False,
        'order_evidence': '',  # 同一時間内も含む順序を確認した情報源
        'history_evidence': '',  # 本人の初回からの全有効対局と確認した根拠
        'features_evidence': '',  # 抽出関数の版と補完前であることの根拠
    },
    'ultra': {
        'manifest_path': ROOT / 'validation_inputs/ultra_manifest.csv',
        'features_path': ROOT / 'validation_inputs/ultra_raw.csv',
        'dan_column': 'dan',  # 保存データがultra_danなら変更
        'raw_features_confirmed': False,
        'full_history_confirmed': False,
        'order_verified': False,
        'order_evidence': '',
        'history_evidence': '',
        'features_evidence': '',
    },
}
# 任意：同一ランタイムにあるDataFrameを明示指定。指定時は該当ファイルを読まない。
# 例 MEMORY_INPUTS = {('shobon', 'features'): micro_features_df}
# manifestも ('shobon', 'manifest') のキーで指定可能。自動推測はしません。
MEMORY_INPUTS = {}

REFERENCE_RESULTS = pd.DataFrame([
    {'source': '基準版・従来ランダムCV', 'metric': 'ROC-AUC', 'value': 0.8340},
    {'source': 'ウルトラ立直・従来外部評価', 'metric': 'Accuracy', 'value': 0.3559},
    {'source': 'ウルトラ立直・従来外部評価', 'metric': 'Balanced Accuracy', 'value': 0.5080},
])
# 将来の学習条件を記録するだけ。今回この設定で学習しません。
MODEL_PLAN = dict(objective='binary', class_weight='balanced', random_state=42,
                  n_estimators=200, learning_rate=0.03, max_depth=5,
                  subsample=0.8, colsample_bytree=0.8, verbosity=-1)
THRESHOLD_PLAN = 0.5
state = {}  # 設定変更時は旧検査結果を無効化
display(REFERENCE_RESULTS)


## セル3：共通関数の定義（学習・予測共通）

既存の定義上の注意：ノーテンは「門前かつ未リーチ」、無筋は実質非現物、良形は牌種数による代理指標。
副露復号・複数リーチへの安全判定にも制約があります。流局ノーテン率はテンパイ率の補数です。
クロス特徴量は試合単位の積・差で、同一局面の条件付き割合ではありません。**今回は変更しません。**

全台帳の通算1501試合目以降を候補とし、段位17も履歴と学習区間の補完平均に含めます。
履歴用には最初の採点候補の前9試合から必要です（通常1492試合目以降）。
特徴量のNaNと「特徴量行そのものの欠落」は区別し、行欠落は平均補完しません。


In [ ]:
BASE_FEATURES = [
    'first_discard_raw_unsafe_rate', 'first_discard_safe_rate',
    'early_noten_push_rate', 'mid_noten_push_rate', 'late_noten_push_rate',
    'las_desperate_push_rate', 'top_conservative_push_rate', 'flat_situation_push_rate',
    'riichi_dora_mean', 'chasing_riichi_good_shape_rate', 'chasing_riichi_low_value_rate',
    'chasing_riichi_rate', 'early_meld_ratio', 'late_meld_ratio', 'meld_dora_ratio',
    'raw_unsafe_discard_rate', 'ryuukyoku_tenpai_rate', 'ryuukyoku_noten_safe_rate',
]
CROSS_FEATURES = ['top_late_noten_push_rate', 'las_early_meld_rate',
                  'early_meld_with_dora_rate', 'chasing_riichi_quality_diff',
                  'flat_first_discard_safe_rate']
FEATURES = BASE_FEATURES + CROSS_FEATURES

class ValidationBlocked(ValueError):
    pass

def require(condition, message):
    if not condition:
        raise ValidationBlocked(message)

def config_signature():
    return json.dumps({'sources': SOURCES, 'window': WINDOW, 'initial': INITIAL_EXCLUDE,
                       'memory_keys': sorted(map(str, MEMORY_INPUTS))},
                      default=str, sort_keys=True, ensure_ascii=False)

def canonical_game_id(value):
    # パス・視点tw・拡張子を除去。同じ対局の別視点も同じIDにする。
    name = str(value).replace('\\', '/').rsplit('/', 1)[-1].lower()
    match = re.fullmatch(r'(\d{10}gm-[0-9a-f]+-[0-9a-f]+-[0-9a-f]+)(?:&tw=[0-3])?(?:\.mjlog)?', name)
    if not match:
        raise ValidationBlocked(f'解釈できない対局ID: {value}')
    return match.group(1)

def load_table(player, kind):
    if (player, kind) in MEMORY_INPUTS:
        df = MEMORY_INPUTS[(player, kind)].copy(deep=True)
        require(isinstance(df, pd.DataFrame), '入力はDataFrameで指定してください')
        origin = 'explicit in-memory DataFrame'
    else:
        path = Path(SOURCES[player][kind + '_path'])
        require(path.is_file(), f'未発見: {path}')
        if path.suffix.lower() == '.csv':
            df = pd.read_csv(path)
        elif path.suffix.lower() == '.parquet':
            df = pd.read_parquet(path)
        else:
            raise ValidationBlocked('CSV/Parquetのみ対応。pickleや牌譜は読みません')
        origin = str(path)
    require(df.columns.is_unique, '列名が重複しています')
    digest = hashlib.sha256(pd.util.hash_pandas_object(df, index=True).values.tobytes()
                            + repr(list(df.columns)).encode()).hexdigest()
    return df, {'player': player, 'kind': kind, 'origin': origin,
                'rows': len(df), 'content_sha256': digest}

def add_cross(base):
    # 元ノートブックの5式をそのまま維持。局面別再集計はしない。
    x = base[BASE_FEATURES].copy()
    x['top_late_noten_push_rate'] = x['top_conservative_push_rate'] * x['late_noten_push_rate']
    x['las_early_meld_rate'] = x['las_desperate_push_rate'] * x['early_meld_ratio']
    x['early_meld_with_dora_rate'] = x['early_meld_ratio'] * x['meld_dora_ratio']
    x['chasing_riichi_quality_diff'] = x['chasing_riichi_good_shape_rate'] - x['chasing_riichi_low_value_rate']
    x['flat_first_discard_safe_rate'] = x['first_discard_safe_rate'] * (1.0 - x['flat_situation_push_rate'])
    return x[FEATURES]

def fit_imputation(training_rows):
    # 明示された学習区間だけ。外部データや検証区間は渡さない。
    require(len(training_rows) > 0, '補完平均の学習対象が空です')
    means = training_rows[BASE_FEATURES].mean()
    require(np.isfinite(means.to_numpy()).all(),
            '学習区間で全欠損の特徴量があります: ' + str(means.index[means.isna()].tolist()))
    return means

def transform_block(rows, means):
    require(list(means.index) == BASE_FEATURES, '補完値の列順が不正です')
    require(np.isfinite(means.to_numpy()).all(), '補完値に非有限値があります')
    require(len(rows) >= WINDOW, '10試合の履歴を作れません')
    require(rows['game_order'].diff().iloc[1:].eq(1).all(), '移動平均区間に対局の欠落があります')
    require(rows['game_id'].is_unique, '移動平均区間の対局IDが重複しています')
    filled = rows[BASE_FEATURES].fillna(means)
    require(np.isfinite(filled.to_numpy()).all(), '補完後に欠損・無限大が残っています')
    # 計算自体は既存のmin_periods=1を維持。ただし10試合未満は採点しない。
    rolled = add_cross(filled).rolling(window=WINDOW, min_periods=1).mean()
    meta = rows[['file_id', 'game_id', 'game_order', 'dan', 'target_label']].copy()
    ids = rows['game_id'].tolist()
    meta['source_ids'] = [tuple(ids[max(0, i-WINDOW+1):i+1]) for i in range(len(ids))]
    meta['full_window'] = [len(v) == WINDOW for v in meta['source_ids']]
    mask = meta['full_window'] & (meta['game_order'] > INITIAL_EXCLUDE) & meta['target_label'].notna()
    return {'X': rolled.loc[mask, FEATURES].reset_index(drop=True),
            'y': meta.loc[mask, 'target_label'].astype(int).reset_index(drop=True),
            'meta': meta.loc[mask].reset_index(drop=True)}

def source_ids(prepared):
    return {gid for window in prepared['meta']['source_ids'] for gid in window}

def fold_boundaries(n):
    # 初期1500試合を除いた実際の連続対局数で50%, 2/3, 5/6, 100%。段位で除外しない。
    remaining = n - INITIAL_EXCLUDE
    require(remaining >= 60, '初期期間後が60試合未満で3分割の履歴を確保できません')
    return [INITIAL_EXCLUDE + int(remaining * fraction) for fraction in (0.5, 2/3, 5/6, 1)]

def make_folds(data):
    boundaries = fold_boundaries(int(data['game_order'].max()))
    plans = []
    for k, (end, next_end) in enumerate(zip(boundaries[:-1], boundaries[1:]), 1):
        # trainには初期期間の末尾9試合の履歴を含める。
        train = data[data['game_order'] <= end].copy()
        validation = data[(data['game_order'] > end) & (data['game_order'] <= next_end)].copy()
        plans.append((k, train, validation))
    return plans

def audit_player(player, manifest, features, cfg):
    problems, tables = [], {}
    require('file_id' in manifest and 'file_id' in features, '両表にfile_idが必要です')
    require('game_order' in manifest and cfg['dan_column'] in manifest,
            '全対局台帳にgame_orderと指定した段位列が必要です')
    m, f = manifest.copy(), features.copy()
    m['game_id'] = m['file_id'].map(canonical_game_id)
    f['game_id'] = f['file_id'].map(canonical_game_id)
    for label, table in [('manifest', m), ('features', f)]:
        duplicate = table[table['game_id'].duplicated(keep=False)]
        tables[label + '_duplicates'] = duplicate
        if len(duplicate):
            problems.append(f'{label}: 正規化対局ID重複 {len(duplicate)}行（自動削除しません）')
    missing_cols = [c for c in BASE_FEATURES if c not in f]
    if missing_cols:
        problems.append('不足特徴量列: ' + str(missing_cols))
    m['game_order'] = pd.to_numeric(m['game_order'], errors='coerce')
    good_order = (m['game_order'].notna().all() and
                  np.array_equal(np.sort(m['game_order'].to_numpy()), np.arange(1, len(m)+1)))
    if not good_order:
        problems.append('game_orderが初回からの1始まり重複なし連番ではありません')
    m = m.sort_values('game_order', kind='stable').reset_index(drop=True)
    m['id_hour'] = pd.to_datetime(m['game_id'].str[:10], format='%Y%m%d%H', errors='coerce')
    if m['id_hour'].isna().any() or not m['id_hour'].is_monotonic_increasing:
        problems.append('対局IDの日時が不正、またはgame_orderと逆転しています')
    tables['same_hour'] = m[m['id_hour'].duplicated(keep=False)][['file_id', 'game_order', 'id_hour']]
    if not (cfg['order_verified'] and str(cfg['order_evidence']).strip()):
        problems.append('厳密な対局順序の保証・根拠が未確認（IDは時単位）')
    if not (cfg['full_history_confirmed'] and str(cfg['history_evidence']).strip()):
        problems.append('初回からの全有効対局台帳の完全性が未確認')
    if not (cfg['raw_features_confirmed'] and str(cfg['features_evidence']).strip()):
        problems.append('特徴量が補完・平滑化前で元定義と同一か未確認')
    m['dan'] = pd.to_numeric(m[cfg['dan_column']], errors='coerce')
    invalid_dan = m['dan'].isna() | ~np.isfinite(m['dan']) | (m['dan'] % 1 != 0) | ~m['dan'].between(0, 20)
    tables['invalid_dan'] = m.loc[invalid_dan, ['file_id', 'game_order', 'dan']]
    if (invalid_dan & (m['game_order'] > INITIAL_EXCLUDE)).any():
        problems.append('採点候補期間に欠損・不正段位があります')
    m['target_label'] = np.select([m['dan'] >= 18, m['dan'] <= 16], [1, 0], default=np.nan)
    # 初回採点1501試合目の窓は1492〜1501試合目。台帳自体は初回から必要。
    needed = m[m['game_order'] >= INITIAL_EXCLUDE - WINDOW + 2].copy()
    tables['missing_games'] = needed[~needed['game_id'].isin(f['game_id'])][['file_id', 'game_id', 'game_order', 'dan']]
    tables['extra_features'] = f.loc[~f['game_id'].isin(m['game_id']), ['file_id', 'game_id']]
    if len(tables['missing_games']):
        problems.append(f"必要な特徴量行が{len(tables['missing_games'])}試合不足（missing_games表）")
    if len(tables['extra_features']):
        problems.append('台帳に存在しない特徴量行があります（別人物・台帳欠落を確認）')
    if len(m) <= INITIAL_EXCLUDE:
        problems.append('初期1500試合を除くと採点候補がありません')
    if not missing_cols:
        numeric = f[BASE_FEATURES].apply(pd.to_numeric, errors='coerce')
        bad_text = f[BASE_FEATURES].notna() & numeric.isna()
        if bad_text.any().any():
            problems.append('特徴量に数値変換不能な値があります')
        if np.isinf(numeric.to_numpy()).any():
            problems.append('特徴量に無限大があります。由来不明のため自動補正しません')
        f[BASE_FEATURES] = numeric
        tables['missingness'] = pd.DataFrame({'feature': BASE_FEATURES,
            'missing': numeric.isna().sum().values, 'missing_rate': numeric.isna().mean().values})
    tables['classes'] = (m.assign(period=np.where(m['game_order'] <= INITIAL_EXCLUDE, '初期・採点外', '採点候補'),
                                  label=m['target_label'].map({0:'弱',1:'強'}).fillna('除外/不明'))
                           .groupby(['period', 'label']).size().rename('games').reset_index())
    tables['overview'] = pd.DataFrame([{'player': player, 'manifest_games':len(m),
        'feature_rows': len(f), 'required_history_games':len(needed),
        'missing_history_games':len(tables['missing_games']), 'same_hour_rows':len(tables['same_hour']),
        'missing_feature_columns':len(missing_cols), 'problems':len(problems)}])
    merged = None
    if not problems:
        merged = needed.merge(f[['game_id'] + BASE_FEATURES], on='game_id', how='left', validate='one_to_one')
        merged = merged.sort_values('game_order').reset_index(drop=True)
    return merged, m, tables, problems

def show_save(name, table, report_dir):
    print(f'\n{name}: {len(table)}行（全件はCSV）')
    display(table.head(30))
    table.to_csv(report_dir / (name + '.csv'), index=False, encoding='utf-8-sig')

def verify_current_audit():
    require(state.get('audit_ok', False), '先にセル4の入力監査を完了してください')
    require(state.get('signature') == config_signature(), '設定が変更されました。セル4から再検査してください')

state = {}  # 関数を変更・再実行した場合も前回結果を無効化
print('共通関数を定義しました。次はセル4の入力監査です。')


## セル4：入力監査・不足対局一覧（実データ読込、学習なし）

両プレイヤーを独立に検査してから停止判定します。未発見のファイルは勝手に探索しません。
manifest自体がない場合、必要試合数は「未確定」です。完全な台帳がある場合に限り不足対局を全件CSV出力します。
別フォルダ・別視点の同一対局も外部分離違反として停止します。自動的な除外は行いません。


In [ ]:
state = {'audit_ok': False, 'folds_ok': False, 'signature': config_signature()}
run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid.uuid4().hex[:8]
report_dir = REPORT_ROOT / run_id
report_dir.mkdir(parents=True, exist_ok=False)
state['report_dir'] = report_dir
prepared, manifests, errors, origins = {}, {}, [], []
for player, cfg in SOURCES.items():
    loaded = {}
    for kind in ('manifest', 'features'):
        try:
            loaded[kind], origin = load_table(player, kind)
            origins.append(origin)
        except (ValueError, OSError, ImportError, KeyError) as exc:
            errors.append({'player':player, 'problem':f'{kind}: {exc}'})
    if len(loaded) != 2:
        continue
    try:
        data, manifest, tables, problems = audit_player(player, loaded['manifest'], loaded['features'], cfg)
        manifests[player] = manifest
        for name, table in tables.items():
            show_save(player + '_' + name, table, report_dir)
        errors.extend({'player':player, 'problem':p} for p in problems)
        if data is not None:
            prepared[player] = data
    except (ValueError, KeyError) as exc:
        errors.append({'player':player, 'problem':str(exc)})

if len(manifests) == 2:
    common = manifests['shobon'][['game_id', 'file_id']].merge(
        manifests['ultra'][['game_id', 'file_id']], on='game_id', suffixes=('_shobon','_ultra'))
    show_save('cross_player_shared_games', common, report_dir)
    if len(common):
        errors.append({'player':'both', 'problem':f'プレイヤー間に同一対局{len(common)}行。分離方針の決定が必要です'})
show_save('input_provenance', pd.DataFrame(origins), report_dir)
show_save('blocking_issues', pd.DataFrame(errors, columns=['player','problem']), report_dir)
REFERENCE_RESULTS.to_csv(report_dir / 'historical_reference_only.csv', index=False)
(report_dir / 'config.json').write_text(json.dumps({
    'sources':SOURCES, 'window':WINDOW, 'initial_exclude':INITIAL_EXCLUDE,
    'model_plan_only':MODEL_PLAN, 'threshold_plan':THRESHOLD_PLAN,
    'pandas':pd.__version__, 'numpy':np.__version__, 'baseline_commit':'4d7b41dae7901b08ef7f865b8591348f2cbed642',
}, default=str, ensure_ascii=False, indent=2), encoding='utf-8')
print('検査表の保存先:', report_dir)
require(not errors and len(prepared) == 2,
        '入力監査で停止。blocking_issues.csvとmissing_games.csvを確認してください。学習・牌譜解析はしていません。')
state.update(audit_ok=True, data=prepared, origins=origins)
print('入力監査合格。次はセル5です。')


## セル5：時系列3分割・共通前処理・元対局ID監査

境界は初期期間後の対局数だけで固定し、段位やスコアで調整しません。
各評価区間の先頭9試合は履歴専用。窓は区間ごとにリセットし、学習との重複をゼロにします。
欠損補完平均はその分割の**学習区間の1501試合目以降（段位17も含む）**だけから算出します。
初期期間の末尾9試合は履歴専用で、平均の推定・学習ラベルには使いません。
評価側が単一クラスの場合はAUC不可と記録します。学習側に両クラスがなければ停止します。
各foldは独立した評価です。過去foldの評価期間が後のfoldの学習期間に入るのは拡大型分割の仕様です。


In [ ]:
state['folds_ok'] = False
state.pop('folds', None)
state.pop('external_ready', None)
verify_current_audit()
folds, fold_rows, fold_errors = [], [], []
for fold_no, train_rows, val_rows in make_folds(state['data']['shobon']):
    record = {'fold':fold_no, 'train_end_order':int(train_rows['game_order'].max()),
              'validation_start_order':int(val_rows['game_order'].min()),
              'validation_end_order':int(val_rows['game_order'].max()),
              'validation_history_only':min(WINDOW-1, len(val_rows))}
    try:
        fitting_rows = train_rows[train_rows['game_order'] > INITIAL_EXCLUDE]
        means = fit_imputation(fitting_rows)
        train = transform_block(train_rows, means)
        validation = transform_block(val_rows, means)
        require(len(train['X']) > 0 and len(validation['X']) > 0, '学習または評価の採点対象が0件です')
        shared = source_ids(train) & source_ids(validation)
        fit_shared = set(fitting_rows['game_id']) & source_ids(validation)
        record.update(train_weak=int(train['y'].eq(0).sum()), train_strong=int(train['y'].eq(1).sum()),
                      validation_weak=int(validation['y'].eq(0).sum()), validation_strong=int(validation['y'].eq(1).sum()),
                      window_overlap=len(shared), imputation_fit_overlap=len(fit_shared),
                      train_both_classes=train['y'].nunique()==2,
                      auc_available=validation['y'].nunique()==2)
        require(not shared and not fit_shared, '元対局IDまたは補完推定データが評価側と重複しています')
        require(train['y'].nunique() == 2, '学習側が単一クラスです。境界を勝手に変更せず停止します')
        means.rename('mean').to_csv(report_dir / f'fold{fold_no}_imputation_means.csv')
        for split_name, block in [('train',train),('validation',validation)]:
            block['meta'].to_csv(report_dir / f'fold{fold_no}_{split_name}_windows.csv', index=False)
        folds.append({'fold':fold_no, 'means':means, 'train':train, 'validation':validation})
        record['status'] = 'OK' if record['auc_available'] else 'OK（評価側単一クラス・AUC不可）'
    except ValidationBlocked as exc:
        record['status'] = 'STOP: ' + str(exc)
        fold_errors.append(str(exc))
    fold_rows.append(record)
show_save('fold_summary', pd.DataFrame(fold_rows), report_dir)
require(not fold_errors and len(folds)==3, '分割監査で停止。学習は実行していません。')
state.update(folds_ok=True, folds=folds)
print('3分割の入力行列をstate["folds"]に作成しました。モデルは未学習です。')


## セル6：外部予測用の入力準備・監査（予測・性能評価なし）

しょぼんn全学習期間からだけ補完平均を算出し、同じ関数で両プレイヤーを変換します。
ウルトラ立直の特徴量・段位を使って平均、モデル、閾値を調整しません。
両人物の初期1500試合・段位17は採点対象外ですが、窓の履歴には残ります。
外部予測値は生成しません。ここまで合格してから、別の指示で学習・評価を実装します。


In [ ]:
state.pop('external_ready', None)
verify_current_audit()
require(state.get('folds_ok', False), 'セル5の3分割監査を先に完了してください')
shobon = state['data']['shobon']
ultra = state['data']['ultra']
final_means = fit_imputation(shobon[shobon['game_order'] > INITIAL_EXCLUDE])
final_train = transform_block(shobon, final_means)
external_input = transform_block(ultra, final_means)
require(final_train['y'].nunique()==2, '最終学習用の入力に両クラスがありません')
require(len(external_input['X']) > 0, '外部の採点候補がありません')
shared = source_ids(final_train) & source_ids(external_input)
require(not shared, '外部用の窓に学習側と同一対局があります')
summary = pd.DataFrame([
    {'dataset':name, 'rows':len(block['X']), 'features':block['X'].shape[1],
     'weak':int(block['y'].eq(0).sum()), 'strong':int(block['y'].eq(1).sum()),
     'strong_ratio':float(block['y'].mean()), 'remaining_na':int(block['X'].isna().sum().sum()),
     'both_classes':block['y'].nunique()==2}
    for name,block in [('shobon_final_training_input',final_train), ('ultra_external_input',external_input)]
])
show_save('external_input_summary_NO_PERFORMANCE', summary, report_dir)
final_means.rename('mean').to_csv(report_dir / 'final_training_only_imputation_means.csv')
for name, block in [('shobon_final',final_train), ('ultra_external',external_input)]:
    block['meta'].to_csv(report_dir / (name + '_windows.csv'), index=False)
    pd.concat([block['meta'][['game_id','game_order']],block['X'],block['y'].rename('target_label')],axis=1).to_csv(
        report_dir / (name + '_prepared_input.csv'), index=False)
(report_dir / 'feature_order.json').write_text(json.dumps(FEATURES, indent=2), encoding='utf-8')
state['external_ready'] = {'means':final_means, 'train':final_train, 'external':external_input}
print('入力準備・監査完了。モデル学習・外部予測・指標計算は未実行です。')
print('保存先:', report_dir)
if external_input['y'].nunique() != 2:
    print('注意：外部採点対象が単一クラスのため、今後ROC-AUCと2クラスのBalanced Accuracyは評価できません。')


## 結果の読み方と次の作業

- `blocking_issues.csv`：止まった理由。`*_missing_games.csv`：追加で必要な対局IDと件数。
- `*_missingness.csv`：元特徴量のNaN。事象非発生の欠損は通常あり得ます。
- `*_same_hour.csv`：IDだけでは順序が決まらない同一時間内の対局。確認根拠が必要です。
- `fold_summary.csv`：両クラス、採点件数、窓・補完データ重複の確認。
- `external_input_summary_NO_PERFORMANCE.csv`：外部入力の件数・比率。性能結果ではありません。

各実行は新しいUTC時刻＋ランダムIDのフォルダへ保存し、既存ファイルを上書きしません。
外部評価は一度結果を見ているため完全な未使用テストではありません。今後の調整には使いません。
時系列窓同士の相関は評価区間内にも残ります。試合数を独立標本数と同一視しません。
同一対局照合は行いますが、本人が対戦相手として出現するかは元牌譜なしでは確認しません。
本実装は実際の連続10試合を厳密に要求します。台帳や順序を保証できない場合、ランダム順や段位除外後の行列で代用しません。
